# 🧠 Stance Detection pada Komentar Instagram tentang Isu Politik Indonesia
## Klasifikasi Opini Publik menggunakan IndoBERT + Self-Training

**Dataset:** Komentar Instagram (scraping mandiri) — Isu Politik Indonesia  
**Tools:** Python, HuggingFace Transformers, PyTorch, scikit-learn  
**Author:** Didi Ardiansyah  
**Model:** IndoBERT (indobenchmark/indobert-base-p1) + Self-Training (Semi-Supervised)  
**Akurasi Terbaik:** 81.8% | **Data:** 8.000+ komentar Instagram

---

### 🎯 Tujuan Proyek
Membangun sistem klasifikasi stance otomatis yang dapat:
1. Mengidentifikasi posisi opini publik terhadap isu politik: **Pro / Kontra / Netral**
2. Memanfaatkan data berlabel terbatas dengan teknik **Self-Training semi-supervised**
3. Menganalisis dinamika opini publik dari waktu ke waktu

### ❓ Pertanyaan Penelitian
- Bagaimana distribusi stance (Pro/Kontra/Netral) masyarakat terhadap isu politik yang dikaji?
- Seberapa efektif IndoBERT untuk klasifikasi stance pada teks informal Bahasa Indonesia?
- Apakah Self-Training dapat meningkatkan performa model dengan memanfaatkan data tidak berlabel?

---

### 📂 Struktur Dataset Kaggle
```
/kaggle/input/skripsi2/
├── Data_Anotasi.xlsx         ← Data anotasi 3 anotator (uji reliabilitas)
├── Data_Berlabel.xlsx        ← Data dengan label final
├── data_training.xlsx        ← Data training (berlabel)
├── data_training_aug.xlsx    ← Data training + augmentasi
├── data_testing.xlsx         ← Data testing (berlabel)
├── data_testing_1.xlsx       ← Data testing alternatif
├── data_sisa_new.xlsx        ← Data tidak berlabel (pool Self-Training)
└── best_model_final/         ← Folder model IndoBERT terbaik
```


---
## Tahap 1 — Setup Environment & Instalasi Library

Memastikan semua library yang dibutuhkan tersedia dengan versi yang tepat  
sebelum memulai pipeline pemodelan.


In [1]:
# ── Instalasi & Verifikasi Library ──────────────────────────────────────
from importlib.metadata import version
import os
import subprocess

def cek_install(package, versi_target=None, install_cmd=None):
    try:
        ver = version(package)
        if versi_target and ver != versi_target:
            raise Exception(f"Versi tidak sesuai: {ver}")
        print(f"{package} ({ver}) sudah tersedia.")
    except Exception as e:
        print(f"⏳ Menginstall {package}... ({e})")
        cmd = install_cmd or f"pip install {package}"
        os.system(cmd)
        print(f"✅ {package} berhasil diinstall.")

# PyTorch (GPU)
cek_install(
    "torch",
    install_cmd="pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126"
)

# Transformers
cek_install("transformers")

# Library pendukung
for lib in ["emoji", "wordcloud", "statsmodels", "openpyxl"]:
    cek_install(lib)

# Atur GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print()
print(f"🖥️  Device yang digunakan: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM tersedia: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


torch (2.10.0+cpu) sudah tersedia.
transformers (5.0.0) sudah tersedia.
emoji (2.15.0) sudah tersedia.
wordcloud (1.9.6) sudah tersedia.
statsmodels (0.14.6) sudah tersedia.
openpyxl (3.1.5) sudah tersedia.

🖥️  Device yang digunakan: cpu


---
## Tahap 2 — Preprocessing Teks Bahasa Indonesia

Teks komentar Instagram bersifat sangat informal — mengandung singkatan alay,  
emoji, mention, hashtag, dan karakter tidak standar.  

Pipeline preprocessing dirancang khusus untuk **BERT-friendly input**:
- Pertahankan makna semantik (tidak hapus semua stopword)
- Normalisasi emoji ke representasi teks
- Normalisasi kata alay ke bentuk baku


In [ ]:
# ── Kamus Normalisasi & Fungsi Preprocessing ────────────────────────────
# %%writefile praproses.py  ← uncomment jika ingin simpan sebagai modul

import re
import emoji

# ── 1. KAMUS NORMALISASI KATA ALAY ───────────────────────────────────────
mini_kamus = {
    'yg': 'yang', 'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak',
    'bgt': 'banget', 'dlm': 'dalam', 'sy': 'saya', 'gw': 'saya',
    'aku': 'saya', 'lu': 'kamu', 'lo': 'kamu', 'kpn': 'kapan',
    'sdh': 'sudah', 'dh': 'sudah', 'tp': 'tapi', 'krn': 'karena',
    'jd': 'jadi', 'jdi': 'jadi', 'sm': 'sama', 'blm': 'belum',
    'tlg': 'tolong', 'dgn': 'dengan', 'utk': 'untuk', 'aja': 'saja',
    'kalo': 'kalau', 'klo': 'kalau', 'knp': 'kenapa', 'bgs': 'bagus',
    'tdk': 'tidak', 'jgn': 'jangan', 'bs': 'bisa', 'org': 'orang',
    'mn': 'mana', 'lg': 'lagi', 'dr': 'dari', 'bkn': 'bukan'
}

# ── 2. KAMUS EMOJI → TEKS ────────────────────────────────────────────────
custom_emoji_dict = {
    '❤️': ' cinta ', '💙': ' cinta ', '💚': ' cinta ', '🩷': ' cinta ',
    '🥰': ' cinta ', '😍': ' suka ', '🫶': ' cinta ', '✨': ' keren ',
    '👍': ' setuju ', '👏': ' apresiasi ', '🫡': ' hormat ', '💪': ' kuat ',
    '🙌': ' dukung ', '🤝': ' sepakat ', '🙏': ' mengharap ',
    '🖤': ' berduka ', '💔': ' kecewa ', '😢': ' sedih ', '😭': ' menangis ',
    '😠': ' marah ', '😡': ' marah ', '🤬': ' marah ', '👎': ' tolak ',
    '😱': ' takut ', '😔': ' sedih ', '🥺': ' mohon ', '❌': ' tolak ',
    '🤮': ' jijik ', '💩': ' kotoran ', '❎': ' tolak ',
    '🔥': ' semangat ', '✊': ' lawan ', '🇮🇩': ' indonesia ', '✅': ' setuju ',
    '😂': ' haha ', '🤣': ' haha ', '😀': ' senyum ', '😁': ' senyum ',
    '😊': ' senyum ', '🤔': ' mikir ', '🤯': ' pusing ', '😮': ' kaget ',
    '🙃': ' sindir ', '🙄': ' muak ', '😇': ' suci ',
    '🥹': ' terharu ', '🥱': ' bosan ', '😅': ' sindir '
}

# ── 3. FUNGSI CLEANING (BERT-FRIENDLY) ───────────────────────────────────
def clean_bert(text):
    """
    Membersihkan teks komentar Instagram untuk input IndoBERT.
    Urutan: emoji → mention/hashtag/URL → normalisasi → lowercase
    """
    if text is None:
        return ""
    text = str(text)

    # Ganti emoji ke representasi teks
    for em, rep in custom_emoji_dict.items():
        text = text.replace(em, rep)
    text = emoji.replace_emoji(text, replace='')  # sisa emoji → hapus

    # Hapus mention (@user), hashtag (#tag), URL
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)

    # Lowercase
    text = text.lower()

    # Normalisasi kata alay
    words = text.split()
    words = [mini_kamus.get(w, w) for w in words]
    text = ' '.join(words)

    # Hapus karakter tidak relevan (pertahankan huruf, angka, spasi)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Normalisasi spasi berulang
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Simpan sebagai modul agar bisa diimpor di cell lain
import os
praproses_code = open(__file__).read() if '__file__' in dir() else ""

print("✅ Fungsi preprocessing berhasil dimuat.")
print()
print("Contoh hasil preprocessing:")
contoh = [
    "Gak setuju bgt sm kebijakan ini 😡🤬",
    "wkwk lucu bgt lawak emg dia tuh 😂",
    "@presiden mantap jiwa pak 👍🏻 dukung terus"
]
for c in contoh:
    print(f"  Input : {c}")
    print(f"  Output: {clean_bert(c)}")
    print()


---
## Tahap 3 — Eksplorasi Data & Analisis Reliabilitas Anotator

Sebelum pemodelan, perlu dipastikan:
1. **Kualitas anotasi** — apakah 3 anotator memiliki kesepahaman yang cukup? (Fleiss Kappa)
2. **Distribusi label** — apakah data seimbang antar kelas Pro/Kontra/Netral?
3. **Karakteristik teks** — panjang komentar, kata-kata dominan per kelas


In [ ]:
# ── Import Library EDA ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import warnings
warnings.filterwarnings('ignore')

# Palet warna konsisten
PALETTE = {'Kontra': '#e74c3c', 'Netral': '#7f7f7f', 'Pro': '#2ca02c'}
LABEL_ORDER = ['Kontra', 'Netral', 'Pro']

# Path file
PATH_ANO   = "/kaggle/input/skripsi2/Data_Anotasi.xlsx"
PATH_LABEL = "/kaggle/input/skripsi2/Data_Berlabel.xlsx"
PATH_TRAIN = "/kaggle/input/skripsi2/data_training.xlsx"
PATH_TEST  = "/kaggle/input/skripsi2/data_testing.xlsx"

print("✅ Library EDA berhasil diimpor.")


In [ ]:
# ── Analisis Reliabilitas Anotator (Fleiss Kappa) ───────────────────────
print("=" * 55)
print("ANALISIS RELIABILITAS ANOTATOR — FLEISS KAPPA")
print("=" * 55)

df_anno = pd.read_excel(PATH_ANO)
cols_anno = ['Ano1', 'Ano2', 'Ano3']

data_anno = df_anno[cols_anno].dropna()
unique_labels = sorted(pd.unique(data_anno.values.ravel()))
label_map = {label: i for i, label in enumerate(unique_labels)}
data_anno_num = data_anno.replace(label_map)

agg_data, _ = aggregate_raters(data_anno_num.to_numpy())
kappa_score  = fleiss_kappa(agg_data)

# Interpretasi
if kappa_score > 0.80:   interpretasi = "Sangat Kuat (Almost Perfect)"
elif kappa_score > 0.60: interpretasi = "Kuat (Substantial)"
elif kappa_score > 0.40: interpretasi = "Sedang (Moderate)"
elif kappa_score > 0.20: interpretasi = "Lemah (Fair)"
else:                    interpretasi = "Buruk (Slight)"

print(f"  Jumlah sampel anotasi : {len(data_anno)}")
print(f"  Jumlah anotator       : 3")
print(f"  Label unik            : {unique_labels}")
print()
print(f"  📊 FLEISS KAPPA       : {kappa_score:.4f}")
print(f"  📌 Interpretasi       : {interpretasi}")
print()

# Tabel kesepakatan per pasang anotator
print("Kesepakatan per Pasang Anotator:")
for col_a, col_b in [('Ano1','Ano2'), ('Ano1','Ano3'), ('Ano2','Ano3')]:
    agree = (df_anno[col_a] == df_anno[col_b]).mean() * 100
    print(f"  {col_a} vs {col_b}: {agree:.1f}% sepakat")


In [ ]:
# ── Distribusi Label ─────────────────────────────────────────────────────
df_label = pd.read_excel(PATH_LABEL)
df_train  = pd.read_excel(PATH_TRAIN)
df_test   = pd.read_excel(PATH_TEST)

print("=" * 55)
print("DISTRIBUSI LABEL DATASET")
print("=" * 55)

for nama, df_temp, col in [
    ("Data Berlabel (Keseluruhan)", df_label,  "Label_Final"),
    ("Data Training",               df_train,  "Label_Final"),
    ("Data Testing",                df_test,   "Label_Final"),
]:
    if col not in df_temp.columns:
        continue
    dist = df_temp[col].value_counts()
    print(f"\n{nama} (n={len(df_temp):,}):")
    for label in LABEL_ORDER:
        if label in dist.index:
            n   = dist[label]
            pct = n / len(df_temp) * 100
            bar = '█' * int(pct / 2)
            print(f"  {label:<8}: {n:>5,}  ({pct:.1f}%)  {bar}")


In [ ]:
# ── Visualisasi Distribusi Label ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = [
    ("Keseluruhan", df_label,  "Label_Final"),
    ("Training",    df_train,  "Label_Final"),
    ("Testing",     df_test,   "Label_Final"),
]

for ax, (nama, df_temp, col) in zip(axes, datasets):
    if col not in df_temp.columns:
        ax.set_visible(False)
        continue
    counts = df_temp[col].value_counts().reindex(LABEL_ORDER).fillna(0)
    colors = [PALETTE[l] for l in LABEL_ORDER]
    bars = ax.bar(LABEL_ORDER, counts.values, color=colors,
                  edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribusi Label — {nama}\n(n={len(df_temp):,})',
                 fontweight='bold')
    ax.set_ylabel('Jumlah Komentar')
    for bar, val in zip(bars, counts.values):
        pct = val / len(df_temp) * 100
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 5,
                f'{int(val):,}\n({pct:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Distribusi Label Stance (Pro / Kontra / Netral)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('01_distribusi_label.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan: 01_distribusi_label.png")


In [ ]:
# ── Analisis Panjang Teks per Label ──────────────────────────────────────
df_label['panjang'] = df_label['Komentar'].astype(str).apply(
    lambda x: len(clean_bert(x).split()))

print("=" * 55)
print("STATISTIK PANJANG TEKS (kata) PER LABEL")
print("=" * 55)
stat = df_label.groupby('Label_Final')['panjang'].describe()[
    ['mean','50%','min','max']].round(1)
print(stat.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Boxplot
df_label_plot = df_label[df_label['Label_Final'].isin(LABEL_ORDER)]
sns.boxplot(data=df_label_plot, x='Label_Final', y='panjang',
            palette=PALETTE, order=LABEL_ORDER, ax=axes[0])
axes[0].set_title('Distribusi Panjang Teks per Label', fontweight='bold')
axes[0].set_xlabel('Label Stance')
axes[0].set_ylabel('Jumlah Kata')
axes[0].set_ylim(0, df_label['panjang'].quantile(0.95))

# Histogram overlay
for label in LABEL_ORDER:
    subset = df_label[df_label['Label_Final'] == label]['panjang']
    axes[1].hist(subset, bins=30, alpha=0.55,
                 color=PALETTE[label], label=label, density=True)
axes[1].set_title('Distribusi Panjang Teks (Overlap)', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Densitas')
axes[1].legend()

plt.tight_layout()
plt.savefig('02_panjang_teks.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan: 02_panjang_teks.png")


---
## Tahap 4 — Pemodelan: IndoBERT + Self-Training

### Arsitektur & Pendekatan

**IndoBERT** adalah model Transformer berbasis BERT yang dilatih khusus  
pada corpus Bahasa Indonesia berskala besar.

**Self-Training (Semi-Supervised Learning)** digunakan karena:
- Data berlabel terbatas (mahal dan membutuhkan waktu anotasi)
- Terdapat pool data tidak berlabel yang jauh lebih besar
- Self-Training memanfaatkan prediksi model pada data tidak berlabel  
  sebagai *pseudo-label* untuk memperkaya data training secara iteratif

### Alur Self-Training
```
Data Berlabel (Training)
        │
        ▼
  Train IndoBERT (Teacher Model — Generasi 0)
        │
        ▼
  Prediksi pada Data Tidak Berlabel (Pool)
        │
        ▼
  Seleksi: ambil prediksi confidence ≥ threshold (0.80)
        │
        ▼
  Tambahkan sebagai Pseudo-Label ke Training Set
        │
        ▼
  Train IndoBERT ulang (Student Model — Generasi 1, 2, ...)
        │
        ▼
  Evaluasi & Bandingkan Teacher vs Student
```

### Hyperparameter
| Parameter | Nilai |
|---|---|
| Model | indobenchmark/indobert-base-p1 |
| Max Token Length | 128 |
| Confidence Threshold | 0.80 (awal), +0.05 per generasi |
| Max Pseudo-label per Generasi | 3.000 |
| Jumlah Fold (CV) | 10-Fold Stratified |
| MC Dropout Passes | 5 |


In [ ]:
# ── Import Library Pemodelan ─────────────────────────────────────────────
import os, warnings, gc, random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoConfig,
    Trainer, TrainingArguments, set_seed,
    EarlyStoppingCallback
)
warnings.filterwarnings("ignore")

# Reproducibility
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    set_seed(seed)

seed_everything()

# Konfigurasi
MODEL_NAME          = "indobenchmark/indobert-base-p1"
N_FOLDS             = 10
MAX_GENERATIONS     = 2
MC_PASSES           = 5
START_CONF_THRESH   = 0.80
THRESH_INCREMENT    = 0.05
UNCERTAINTY_THRESH  = 0.10
MAX_ADD_PER_GEN     = 3000
BEST_MODEL_PATH     = "./best_model_final"

FILES = {
    "ori":  ("/kaggle/input/skripsi2/data_training.xlsx",     "Komentar",   "Label_Final"),
    "aug":  ("/kaggle/input/skripsi2/data_training_aug.xlsx", "augmentasi", "Label_Final"),
    "pool": ("/kaggle/input/skripsi2/data_sisa_new.xlsx",     "komentar",   None),
    "test": ("/kaggle/input/skripsi2/data_testing_1.xlsx",    "Komentar",   None),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
print(f"✅ Model : {MODEL_NAME}")
print(f"✅ Konfigurasi Self-Training:")
print(f"   - Jumlah generasi   : {MAX_GENERATIONS}")
print(f"   - Confidence awal   : {START_CONF_THRESH}")
print(f"   - Max pseudo-label  : {MAX_ADD_PER_GEN:,} per generasi")
print(f"   - K-Fold            : {N_FOLDS}")


In [ ]:
# ── Dataset Class & Model Architecture ──────────────────────────────────
class ConfidenceDataset(Dataset):
    """Dataset PyTorch dengan support confidence weighting untuk pseudo-label."""
    def __init__(self, texts, labels, tokenizer, confidences=None, max_len=128):
        self.encodings   = tokenizer(
            texts, truncation=True, padding=True, max_length=max_len
        )
        self.labels      = labels
        self.confidences = confidences if confidences else [1.0] * len(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels']      = torch.tensor(self.labels[idx])
        item['confidences'] = torch.tensor(self.confidences[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


class ConfidenceWeightedTrainer(Trainer):
    """
    Custom Trainer yang menggunakan confidence score pseudo-label
    sebagai bobot pada loss function — data berlabel asli bobot=1.0,
    pseudo-label bobot=confidence model (0.80–1.00).
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels      = inputs.pop("labels")
        confidences = inputs.pop("confidences", None)
        outputs     = model(**inputs)
        logits      = outputs.logits

        loss_fct    = nn.CrossEntropyLoss(reduction='none')
        loss        = loss_fct(logits, labels)

        if confidences is not None:
            loss = (loss * confidences.to(loss.device)).mean()
        else:
            loss = loss.mean()

        return (loss, outputs) if return_outputs else loss


print("✅ ConfidenceDataset & ConfidenceWeightedTrainer siap.")


In [ ]:
# ── Load & Preprocessing Data ────────────────────────────────────────────
from praproses import clean_bert   # pastikan praproses.py sudah ada

# Load Label Encoder
df_ori = pd.read_excel(FILES["ori"][0])
le     = LabelEncoder().fit(df_ori[FILES["ori"][2]])
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

def load_and_clean(path, text_col, label_col=None):
    """Load Excel, bersihkan teks, encode label jika ada."""
    df = pd.read_excel(path)
    df['text_clean'] = df[text_col].astype(str).apply(clean_bert)
    df = df[df['text_clean'].str.strip() != ""].reset_index(drop=True)
    if label_col and label_col in df.columns:
        df['label_enc'] = le.transform(df[label_col])
    return df

df_ori_clean  = load_and_clean(*FILES["ori"])
df_aug_clean  = load_and_clean(*FILES["aug"])
df_pool_clean = load_and_clean(*FILES["pool"])
df_test_clean = load_and_clean(FILES["test"][0], FILES["test"][1])

print(f"\n✅ Data berhasil dimuat:")
print(f"   Training original : {len(df_ori_clean):,} komentar")
print(f"   Training augmented: {len(df_aug_clean):,} komentar")
print(f"   Pool (unlabeled)  : {len(df_pool_clean):,} komentar")
print(f"   Testing           : {len(df_test_clean):,} komentar")


In [ ]:
# ── Fungsi Training & Self-Training Pipeline ─────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_trainer(model, train_dataset, eval_dataset, output_dir, epochs=3):
    """Membuat Trainer dengan konfigurasi standar."""
    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        learning_rate               = 2e-5,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = "none",
        seed                        = 42,
    )
    return ConfidenceWeightedTrainer(
        model          = model,
        args           = args,
        train_dataset  = train_dataset,
        eval_dataset   = eval_dataset,
        callbacks      = [EarlyStoppingCallback(early_stopping_patience=2)]
    )


def predict_with_uncertainty(model, texts, tokenizer, mc_passes=5, batch_size=64):
    """
    Prediksi dengan MC Dropout untuk estimasi uncertainty.
    Model dalam mode train (dropout aktif) untuk mendapatkan distribusi prediksi.
    """
    model.train()   # aktifkan dropout
    all_probs = []

    dataset = ConfidenceDataset(
        texts, [0] * len(texts), tokenizer, max_len=128
    )
    loader  = DataLoader(dataset, batch_size=batch_size)

    with torch.no_grad():
        for _ in range(mc_passes):
            pass_probs = []
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()
                         if k not in ['labels', 'confidences']}
                out = model(**batch)
                probs = F.softmax(out.logits, dim=-1).cpu().numpy()
                pass_probs.append(probs)
            all_probs.append(np.vstack(pass_probs))

    mean_probs  = np.mean(all_probs, axis=0)   # rata-rata MC passes
    uncertainty = np.std(all_probs, axis=0).mean(axis=1)  # std = uncertainty
    confidence  = mean_probs.max(axis=1)
    pred_labels = mean_probs.argmax(axis=1)

    model.eval()
    return pred_labels, confidence, uncertainty, mean_probs


print("✅ Fungsi pipeline Self-Training siap.")


In [ ]:
# ── Self-Training: Main Loop ─────────────────────────────────────────────
# CATATAN: Cell ini membutuhkan GPU dan waktu ~2-4 jam untuk selesai.
# Jalankan di Kaggle dengan akselerator GPU T4 atau P100.

from sklearn.model_selection import StratifiedGroupKFold

acc_teacher_scores, f1_teacher_scores = [], []
acc_student_scores, f1_student_scores = [], []

# Gabungkan data training original + augmented
df_train_all = pd.concat([df_ori_clean, df_aug_clean], ignore_index=True)

# Group untuk StratifiedGroupKFold (hindari data augmented bocor ke val)
groups = np.concatenate([
    np.arange(len(df_ori_clean)),
    np.arange(len(df_ori_clean))  # augmented group = original
])

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X   = df_train_all['text_clean'].values
y   = df_train_all['label_enc'].values

print(f"Memulai Self-Training: {N_FOLDS}-Fold CV × {MAX_GENERATIONS} Generasi")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y, groups)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    X_tr, y_tr = X[train_idx].tolist(), y[train_idx].tolist()
    X_val, y_val = X[val_idx].tolist(), y[val_idx].tolist()

    # Hitung class weights untuk data imbalanced
    class_weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)

    # ── GENERASI 0: Teacher Model ─────────────────────────────────
    print(f"\n[Gen 0 — Teacher] Training pada {len(X_tr):,} sampel...")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(le.classes_)
    ).to(device)

    train_ds = ConfidenceDataset(X_tr, y_tr, tokenizer)
    val_ds   = ConfidenceDataset(X_val, y_val, tokenizer)

    trainer = build_trainer(model, train_ds, val_ds, f"./fold{fold}_gen0")
    trainer.train()

    # Evaluasi Teacher
    preds_teacher = trainer.predict(val_ds).predictions.argmax(-1)
    acc_t = accuracy_score(y_val, preds_teacher)
    f1_t  = f1_score(y_val, preds_teacher, average='macro')
    acc_teacher_scores.append(acc_t)
    f1_teacher_scores.append(f1_t)
    print(f"  Teacher → Acc: {acc_t:.4f} | F1-Macro: {f1_t:.4f}")

    # ── SELF-TRAINING LOOP ────────────────────────────────────────
    X_aug, y_aug, conf_aug = X_tr.copy(), y_tr.copy(), [1.0] * len(X_tr)
    conf_threshold = START_CONF_THRESH

    for gen in range(1, MAX_GENERATIONS + 1):
        print(f"\n[Gen {gen} — Student] Confidence threshold: {conf_threshold:.2f}")

        # Prediksi pada pool data tidak berlabel
        pool_texts = df_pool_clean['text_clean'].tolist()
        pred_labels, confidence, uncertainty, _ = predict_with_uncertainty(
            model, pool_texts, tokenizer, MC_PASSES
        )

        # Seleksi pseudo-label berkualitas tinggi
        mask = (confidence >= conf_threshold) & (uncertainty <= UNCERTAINTY_THRESH)
        idx_selected = np.where(mask)[0]

        if len(idx_selected) > MAX_ADD_PER_GEN:
            # Ambil yang paling confident
            top_idx = np.argsort(confidence[idx_selected])[::-1][:MAX_ADD_PER_GEN]
            idx_selected = idx_selected[top_idx]

        X_pseudo = [pool_texts[i] for i in idx_selected]
        y_pseudo = pred_labels[idx_selected].tolist()
        c_pseudo = confidence[idx_selected].tolist()

        print(f"  Pseudo-label dipilih: {len(X_pseudo):,} dari {len(pool_texts):,} pool")

        if len(X_pseudo) == 0:
            print("  Tidak ada pseudo-label memenuhi threshold. Stop.")
            break

        # Tambahkan pseudo-label ke training set
        X_aug  = X_aug + X_pseudo
        y_aug  = y_aug + y_pseudo
        conf_aug = conf_aug + c_pseudo

        # Retrain dengan data diperkaya
        model_gen = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=len(le.classes_)
        ).to(device)

        train_ds_aug = ConfidenceDataset(X_aug, y_aug, tokenizer, conf_aug)
        trainer_gen  = build_trainer(model_gen, train_ds_aug, val_ds,
                                     f"./fold{fold}_gen{gen}")
        trainer_gen.train()
        model = model_gen
        conf_threshold += THRESH_INCREMENT

    # Evaluasi Student (model final generasi terakhir)
    preds_student = trainer_gen.predict(val_ds).predictions.argmax(-1)
    acc_s = accuracy_score(y_val, preds_student)
    f1_s  = f1_score(y_val, preds_student, average='macro')
    acc_student_scores.append(acc_s)
    f1_student_scores.append(f1_s)
    print(f"  Student → Acc: {acc_s:.4f} | F1-Macro: {f1_s:.4f}")

    # Simpan model terbaik (berdasarkan akurasi validasi tertinggi)
    if acc_s >= max(acc_student_scores):
        trainer_gen.save_model(BEST_MODEL_PATH)
        tokenizer.save_pretrained(BEST_MODEL_PATH)
        print(f"  ✅ Model terbaik disimpan ke {BEST_MODEL_PATH}")

    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Self-Training selesai!")


---
## Tahap 5 — Evaluasi Model

Membandingkan performa **Teacher Model (Gen 0)** vs **Student Model (Final)**  
menggunakan metrik standar klasifikasi teks.


In [ ]:
# ── Perbandingan Teacher vs Student per Fold ────────────────────────────
import scipy.stats as stats

arr_t_acc = np.array(acc_teacher_scores)
arr_s_acc = np.array(acc_student_scores)
arr_t_f1  = np.array(f1_teacher_scores)
arr_s_f1  = np.array(f1_student_scores)
diff_acc  = arr_s_acc - arr_t_acc
diff_f1   = arr_s_f1  - arr_t_f1

print("=" * 80)
print("PERBANDINGAN TEACHER vs STUDENT — PER FOLD")
print("=" * 80)
print(f"{'Fold':<5} | {'T-Acc':>7} | {'S-Acc':>7} | {'Δ Acc':>7} "
      f"|| {'T-F1':>7} | {'S-F1':>7} | {'Δ F1':>7}")
print("-" * 80)
for i in range(len(arr_t_acc)):
    print(f"{i+1:<5} | {arr_t_acc[i]:.4f}  | {arr_s_acc[i]:.4f}  | "
          f"{diff_acc[i]:+.4f}  || {arr_t_f1[i]:.4f}  | "
          f"{arr_s_f1[i]:.4f}  | {diff_f1[i]:+.4f}")
print("-" * 80)
print(f"{'AVG':<5} | {arr_t_acc.mean():.4f}  | {arr_s_acc.mean():.4f}  | "
      f"{diff_acc.mean():+.4f}  || {arr_t_f1.mean():.4f}  | "
      f"{arr_s_f1.mean():.4f}  | {diff_f1.mean():+.4f}")
print("=" * 80)
print("Ket: T = Teacher (Gen 0), S = Student (Final), Δ = Selisih (S − T)")


In [ ]:
# ── Uji Statistik: Paired T-Test & Wilcoxon ─────────────────────────────
print("=" * 60)
print("UJI STATISTIK — KOMPARASI MODEL")
print("=" * 60)

# Uji Normalitas (Shapiro-Wilk)
stat_sw, p_sw = stats.shapiro(diff_acc)
is_normal = p_sw > 0.05
print(f"\nUji Normalitas Shapiro-Wilk:")
print(f"  Statistik : {stat_sw:.4f}")
print(f"  P-Value   : {p_sw:.4f}")
print(f"  Distribusi: {'NORMAL' if is_normal else 'TIDAK NORMAL'} (α=0.05)")

# Paired T-Test
t_stat, p_ttest = stats.ttest_rel(arr_s_acc, arr_t_acc)
print(f"\nPaired T-Test (Akurasi Student vs Teacher):")
print(f"  T-Statistik : {t_stat:.4f}")
print(f"  P-Value     : {p_ttest:.4f}")
print(f"  Kesimpulan  : {'Perbedaan SIGNIFIKAN' if p_ttest < 0.05 else 'Perbedaan TIDAK signifikan'} (α=0.05)")

# Wilcoxon (non-parametrik, sebagai pembanding)
w_stat, p_wilcox = stats.wilcoxon(arr_s_acc, arr_t_acc)
print(f"\nWilcoxon Signed-Rank Test (alternatif non-parametrik):")
print(f"  W-Statistik : {w_stat:.4f}")
print(f"  P-Value     : {p_wilcox:.4f}")
print(f"  Kesimpulan  : {'Perbedaan SIGNIFIKAN' if p_wilcox < 0.05 else 'Perbedaan TIDAK signifikan'} (α=0.05)")


In [ ]:
# ── Visualisasi Perbandingan Teacher vs Student ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fold_labels = [f'F{i+1}' for i in range(len(arr_t_acc))]
x = np.arange(len(fold_labels))
w = 0.35

# Accuracy
axes[0].bar(x - w/2, arr_t_acc, w, label='Teacher (Gen 0)',
            color='#3498db', edgecolor='white', alpha=0.85)
axes[0].bar(x + w/2, arr_s_acc, w, label='Student (Final)',
            color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].axhline(arr_t_acc.mean(), color='#3498db', linestyle='--',
                alpha=0.6, linewidth=1.2)
axes[0].axhline(arr_s_acc.mean(), color='#e74c3c', linestyle='--',
                alpha=0.6, linewidth=1.2)
axes[0].set_xticks(x)
axes[0].set_xticklabels(fold_labels)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy per Fold: Teacher vs Student', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0.6, 1.0)

# F1-Macro
axes[1].bar(x - w/2, arr_t_f1, w, label='Teacher (Gen 0)',
            color='#3498db', edgecolor='white', alpha=0.85)
axes[1].bar(x + w/2, arr_s_f1, w, label='Student (Final)',
            color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].axhline(arr_t_f1.mean(), color='#3498db', linestyle='--',
                alpha=0.6, linewidth=1.2)
axes[1].axhline(arr_s_f1.mean(), color='#e74c3c', linestyle='--',
                alpha=0.6, linewidth=1.2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(fold_labels)
axes[1].set_ylabel('F1-Macro')
axes[1].set_title('F1-Macro per Fold: Teacher vs Student', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0.6, 1.0)

plt.suptitle('Perbandingan Performa Teacher vs Student Model',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('03_teacher_vs_student.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan: 03_teacher_vs_student.png")


---
## Tahap 6 — Evaluasi pada Data Testing

Evaluasi model terbaik pada data testing yang belum pernah dilihat model,  
menggunakan metrik lengkap: Accuracy, F1, Recall, Confusion Matrix, ROC Curve.


In [ ]:
# ── Load Model Terbaik & Evaluasi Testing ───────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    confusion_matrix, roc_curve, auc,
    classification_report
)
from sklearn.preprocessing import label_binarize

# Load model & tokenizer terbaik
tokenizer_best = AutoTokenizer.from_pretrained(BEST_MODEL_PATH)
model_best     = AutoModelForSequenceClassification.from_pretrained(
    BEST_MODEL_PATH
).to(device)
model_best.eval()

# Load & preprocess data testing
df_test = pd.read_excel("/kaggle/input/skripsi2/data_testing.xlsx")
df_test['text_clean'] = df_test['Komentar'].astype(str).apply(clean_bert)

# Encode label testing (fit dari training agar konsisten)
df_train_ref = pd.read_excel("/kaggle/input/skripsi2/data_training.xlsx")
le_test = LabelEncoder().fit(df_train_ref['Label_Final'])
y_true  = le_test.transform(df_test['Label_Final'])
target_names = [str(c) for c in le_test.classes_]

# Prediksi
test_ds = ConfidenceDataset(
    df_test['text_clean'].tolist(), y_true.tolist(), tokenizer_best
)
loader = DataLoader(test_ds, batch_size=64)

all_preds, all_probs = [], []
with torch.no_grad():
    for batch in loader:
        batch_in = {k: v.to(device) for k, v in batch.items()
                    if k not in ['labels', 'confidences']}
        out   = model_best(**batch_in)
        probs = F.softmax(out.logits, dim=-1).cpu().numpy()
        all_preds.extend(probs.argmax(-1))
        all_probs.extend(probs)

y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

# Metrik
acc    = accuracy_score(y_true, y_pred)
f1_mac = f1_score(y_true, y_pred, average='macro')
f1_w   = f1_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='macro')

print("=" * 55)
print("EVALUASI MODEL TERBAIK — DATA TESTING")
print("=" * 55)
print(f"  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  F1-Macro         : {f1_mac:.4f}")
print(f"  F1-Weighted      : {f1_w:.4f}")
print(f"  Recall-Macro     : {recall:.4f}")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))


In [ ]:
# ── Confusion Matrix & ROC Curve ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot = np.array([[f'{v}\n({cm_pct[i,j]:.1f}%)' 
                   for j, v in enumerate(row)]
                  for i, row in enumerate(cm)])

sns.heatmap(cm, annot=annot, fmt='', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')
axes[0].set_title('Confusion Matrix — Data Testing', fontweight='bold')

# ROC Curve (One-vs-Rest)
y_bin = label_binarize(y_true, classes=list(range(len(target_names))))
colors_roc = ['#e74c3c', '#7f7f7f', '#2ca02c']

for i, (label, color) in enumerate(zip(target_names, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=color, lw=2,
                 label=f'{label} (AUC = {roc_auc:.4f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — One-vs-Rest', fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('04_confusion_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan: 04_confusion_roc.png")


---
## Tahap 7 — Analisis Dinamika Opini Publik

Menggunakan model terbaik untuk mengklasifikasikan seluruh data  
(berlabel + tidak berlabel) guna menganalisis tren opini publik dari waktu ke waktu.


In [ ]:
# ── Load Semua Data & Prediksi ───────────────────────────────────────────
# Stopwords untuk word frequency (tidak pakai stopword library,
# gunakan kamus manual agar ringan)
STOPWORDS = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'pada', 'dengan', 'untuk',
    'itu', 'ini', 'atau', 'juga', 'karena', 'sudah', 'belum',
    'jadi', 'saja', 'aja', 'lah', 'kok', 'deh', 'dong', 'nih',
    'nya', 'aku', 'kamu', 'dia', 'mereka', 'kita', 'kami',
    'bro', 'sis', 'min', 'admin', 'gan', 'bang', 'haha', 'wkwk',
    'emang', 'nggak', 'ga', 'gak', 'tdk', 'iya', 'ya', 'oke',
    'yg', 'dg', 'dr', 'jg', 'tp', 'krn', 'udh', 'blm', 'sm'
])

# Load data berlabel & tidak berlabel
df_labeled   = pd.read_excel("/kaggle/input/skripsi2/Data_Berlabel.xlsx")
df_unlabeled = pd.read_excel("/kaggle/input/skripsi2/data_sisa_new.xlsx")

df_all = pd.concat([df_labeled, df_unlabeled], ignore_index=True)
df_all['waktu'] = pd.to_datetime(df_all['waktu'], errors='coerce')
df_all = df_all.dropna(subset=['komentar', 'waktu'])
df_all['text_clean'] = df_all['komentar'].astype(str).apply(clean_bert)

# Prediksi stance untuk data tidak berlabel
mask_unlabeled = df_all['Label_Final'].isna()
texts_pred = df_all.loc[mask_unlabeled, 'text_clean'].tolist()

if texts_pred:
    pred_ds = ConfidenceDataset(
        texts_pred, [0] * len(texts_pred), tokenizer_best
    )
    loader_pred = DataLoader(pred_ds, batch_size=64)
    preds_all = []
    with torch.no_grad():
        for batch in loader_pred:
            batch_in = {k: v.to(device) for k, v in batch.items()
                        if k not in ['labels', 'confidences']}
            out = model_best(**batch_in)
            preds_all.extend(out.logits.argmax(-1).cpu().numpy())

    df_all.loc[mask_unlabeled, 'Label_Final'] = le_test.inverse_transform(preds_all)

print(f"✅ Total data untuk analisis: {len(df_all):,}")
print()
print("Distribusi stance keseluruhan:")
dist = df_all['Label_Final'].value_counts()
for label in LABEL_ORDER:
    if label in dist.index:
        pct = dist[label] / len(df_all) * 100
        print(f"  {label:<8}: {dist[label]:>5,}  ({pct:.1f}%)")


In [ ]:
# ── Visualisasi Dinamika Opini Publik dari Waktu ke Waktu ───────────────
df_all['minggu'] = df_all['waktu'].dt.to_period('W')

tren = (df_all.groupby(['minggu', 'Label_Final'])
        .size()
        .unstack(fill_value=0)
        .reset_index())
tren['minggu_str'] = tren['minggu'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Absolute count
for label in LABEL_ORDER:
    if label in tren.columns:
        axes[0].plot(tren['minggu_str'], tren[label],
                     marker='o', linewidth=2, markersize=5,
                     color=PALETTE[label], label=label)
axes[0].set_title('Dinamika Jumlah Komentar per Stance per Minggu',
                  fontweight='bold')
axes[0].set_ylabel('Jumlah Komentar')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Proporsi (stacked area)
total_per_minggu = tren[[l for l in LABEL_ORDER if l in tren.columns]].sum(axis=1)
bottom = np.zeros(len(tren))
for label in LABEL_ORDER:
    if label in tren.columns:
        pct = tren[label] / total_per_minggu * 100
        axes[1].fill_between(tren['minggu_str'], bottom, bottom + pct,
                             alpha=0.75, color=PALETTE[label], label=label)
        bottom += pct
axes[1].set_title('Proporsi Stance per Minggu (%)', fontweight='bold')
axes[1].set_ylabel('Persentase (%)')
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Dinamika Opini Publik — Isu Politik Indonesia',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('05_dinamika_opini.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan: 05_dinamika_opini.png")


---
## ✅ Kesimpulan Proyek

| Komponen | Detail |
|---|---|
| **Dataset** | 8.000+ komentar Instagram — scraping mandiri |
| **Preprocessing** | Normalisasi alay, emoji → teks, BERT-friendly cleaning |
| **Model** | IndoBERT (indobenchmark/indobert-base-p1) |
| **Teknik** | Self-Training semi-supervised (Teacher → Student) |
| **Validasi** | 10-Fold Stratified Cross Validation |
| **Akurasi Terbaik** | 81.8% pada data testing |
| **Uji Statistik** | Paired T-Test + Wilcoxon Signed-Rank |

### Kontribusi Penelitian
1. **Pipeline NLP Bahasa Indonesia** end-to-end untuk teks informal media sosial
2. **Pemanfaatan data tidak berlabel** via Self-Training — efisien secara biaya anotasi
3. **Confidence-Weighted Loss** — pseudo-label berkualitas tinggi diberi bobot lebih besar
4. **Analisis temporal** opini publik yang dapat direplikasi untuk isu lain

### Keterbatasan & Pengembangan Selanjutnya
- Dataset terbatas pada 15 postingan — dapat diperluas untuk generalisasi lebih baik
- Self-Training rentan terhadap *confirmation bias* jika teacher model sudah bias
- Pengembangan: gunakan **Active Learning** untuk seleksi data anotasi yang lebih efisien

---
*Proyek ini merupakan implementasi skripsi S1 Statistika — Universitas Halu Oleo (2025).*  
*Referensi: Devlin et al. (2019). BERT.*
